# Notebook for measuring runtime of Hashing, bucketing and similarity value computation 

In [ ]:
import os
import sys

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root f^ound: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from utils.helpers.measure_similarities import *


# Code for running several combinations of the runtime parameters

# Disk

## Setup

In [ ]:
MEASURE="disk_dtw_cy"
CITY="rome"
DIAMETER_LIST = [1.5]
LAYERS_LIST = [1]
DISKS_LIST = [20]
PARALLEL_JOBS = 24
DATA_SIZE = [50]
ITERATIONS = 1

#Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = True

#Filenames
if TRUE_TRAJECTORIES:
    file_name = f"runtimes_{CITY}_disk_d{min(DIAMETER_LIST)}-{max(DIAMETER_LIST)}_l{min(LAYERS_LIST)}-{max(LAYERS_LIST)}_nd{min(DISKS_LIST)}-{max(DISKS_LIST)}_sz{DATA_SIZE}_multiple_configs_true.csv"
else:
    file_name = f"runtimes_{CITY}_disk_d{min(DIAMETER_LIST)}-{max(DIAMETER_LIST)}_l{min(LAYERS_LIST)}-{max(LAYERS_LIST)}_nd{min(DISKS_LIST)}-{max(DISKS_LIST)}_sz{DATA_SIZE}_multiple_configs.csv"

output_path = f"../../../results_hashed/runtimes/disk/{CITY}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

In [14]:
import itertools
import pandas as pd
import os

# Initialize DataFrame to store results
results = pd.DataFrame()

# Generate all combinations of parameters
param_combinations = list(itertools.product(DIAMETER_LIST, LAYERS_LIST, DISKS_LIST, DATA_SIZE))

first_write = True

# Iterate over each combination and run the function
for diameter, layers, disks, data_size in param_combinations:
    print(f"Running for Diameter: {diameter}, Layers: {layers}, Disks: {disks}, Data Size: {data_size}")


    if TRUE_TRAJECTORIES:
        df_result = compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(
            measure=MEASURE,
            city=CITY,
            diameter=diameter,
            layers=layers,
            disks=disks,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )
    else:
        df_result = compute_hashed_similarity_runtimes_with_bucketing(
            measure=MEASURE,
            city=CITY,
            diameter=diameter,
            layers=layers,
            disks=disks,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )

    # Add parameters to result DataFrame
    df_result["City"] = CITY
    df_result["Measure"] = MEASURE
    df_result["Diameter"] = diameter
    df_result["Layers"] = layers
    df_result["Disks"] = disks
    df_result["Size"] = data_size

    # Define the desired column order
    desired_order = ["City", "Measure", "Diameter", "Layers", "Disks","Size",
                    "Average Similarity Computation Time (Seconds)", 
                    "Average Hash Generation Time (Seconds)", 
                    "Average Bucket Distribution Time (Seconds)", "Total time (Seconds)",]

    # Reorder columns
    df_result = df_result[desired_order]

    # Save the DataFrame to a CSV file
    df_result.to_csv(output_path, mode='a', header=first_write, index=False)
    first_write = False

Running for Diameter: 1.5, Layers: 1, Disks: 20, Data Size: 50
Iteration 1/1
Computing grid_dtw_cy for rome with 24 jobs - Iteration 1/1

Final Runtime Statistics:
   Size Average Similarity Computation Time (Seconds)  \
0    50                                         1.547   

  Average Hash Generation Time (Seconds)  \
0                                  0.120   

  Average Bucket Distribution Time (Seconds) Total time (Seconds)  
0                                      0.001                1.668  


# Grid

## Setup

In [15]:
import numpy as np

MEASURE="grid_dtw_cy"
CITY="rome"
RESOLUTION_LIST = [0.3, 0.5] 
LAYERS_LIST = LAYERS_VALUES = [1]
PARALLEL_JOBS = 8
DATA_SIZE_LIST = [50]
ITERATIONS = 1

#Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = True

In [17]:
import itertools
import pandas as pd
import os

# Initialize DataFrame to store results
results = pd.DataFrame()

# Generate all combinations of parameters
param_combinations = list(itertools.product(RESOLUTION_LIST, LAYERS_LIST, DATA_SIZE_LIST))

# Iterate over each combination and run the function
for resolution, layers, data_size in param_combinations:
    print(f"Running for Resolution: {resolution}, Layers: {layers}, Data Size: {data_size}")

    if TRUE_TRAJECTORIES:
        df_result = compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(
            measure=MEASURE,
            city=CITY,
            res=resolution,  # Grid uses resolution instead of diameter
            layers=layers,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )
    else:
        df_result = compute_hashed_similarity_runtimes_with_bucketing(
            measure=MEASURE,
            city=CITY,
            res=resolution,  # Grid uses resolution instead of diameter
            layers=layers,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )

    # Add parameters to result DataFrame
    df_result["Resolution"] = resolution
    df_result["Layers"] = layers
    df_result["Data Size"] = data_size

    # Define the desired column order
    desired_order = [
        "Data Size", "Resolution", "Layers",
        "Average Similarity Computation Time (Seconds)", 
        "Average Hash Generation Time (Seconds)", 
        "Average Bucket Distribution Time (Seconds)",
        "Total time (Seconds)",
    ]

    # Reorder columns
    df_result = df_result[desired_order]

    # Append to final results DataFrame
    results = pd.concat([results, df_result], ignore_index=True)

# Save final results to a CSV file

if TRUE_TRAJECTORIES:
    file_name = f"runtimes_{CITY}_grid_res{min(RESOLUTION_LIST)}-{max(RESOLUTION_LIST)}_l{min(LAYERS_LIST)}-{max(LAYERS_LIST)}_sz{DATA_SIZE}_multiple_configs_true.csv"
else:
    file_name = f"runtimes_{CITY}_grid_res{min(RESOLUTION_LIST)}-{max(RESOLUTION_LIST)}_l{min(LAYERS_LIST)}-{max(LAYERS_LIST)}_sz{DATA_SIZE}_multiple_configs.csv"

output_path = f"../../../results_hashed/runtimes/grid/{CITY}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

results.to_csv(output_path, index=False)

# Display final results
print("\nFinal Results:")
display(results)


Running for Resolution: 0.3, Layers: 1, Data Size: 50
Iteration 1/1
Computing grid_dtw_cy for rome with 8 jobs - Iteration 1/1

Final Runtime Statistics:
   Size Average Similarity Computation Time (Seconds)  \
0    50                                         1.469   

  Average Hash Generation Time (Seconds)  \
0                                  0.044   

  Average Bucket Distribution Time (Seconds) Total time (Seconds)  
0                                      0.001                1.514  
Running for Resolution: 0.5, Layers: 1, Data Size: 50
Iteration 1/1
Computing grid_dtw_cy for rome with 8 jobs - Iteration 1/1

Final Runtime Statistics:
   Size Average Similarity Computation Time (Seconds)  \
0    50                                         1.409   

  Average Hash Generation Time (Seconds)  \
0                                  0.042   

  Average Bucket Distribution Time (Seconds) Total time (Seconds)  
0                                      0.001                1.452  

Final Resul

,Data Size,Resolution,Layers,Average Similarity Computation Time (Seconds),Average Hash Generation Time (Seconds),Average Bucket Distribution Time (Seconds),Total time (Seconds)
0,50,0.3,1,1.469,0.044,0.001,1.514
1,50,0.5,1,1.409,0.042,0.001,1.452
